# Issue #78: fixed-protocol Qwen benchmark

This notebook reproduces the published unquantized FP16 comparison on a Google Colab Tesla T4. It installs Hashformers directly from the checked-out Git revision used by the recorded run, keeps generated artifacts outside the checkout so provenance remains clean, and runs each model in a separate process. Start with a fresh GPU runtime.

In [ ]:
!git clone https://github.com/ruanchaves/hashformers.git /content/hashformers
%cd /content/hashformers
!git checkout b30e66e163bb5ac9d43da23edd725eda7353adf3
!python -m pip install -q 'transformers>=4.51,<6' 'accelerate>=1'
!python -m pip install -q --no-deps -e /content/hashformers
!nvidia-smi

import subprocess

assert not subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd='/content/hashformers', text=True
).strip(), 'Benchmark checkout must be clean'

In [ ]:
!python scripts/qwen_benchmark.py run \
  --model qwen3 \
  --manifest benchmarks/qwen/samples.jsonl \
  --device cuda:0 \
  --precision float16 \
  --quantization none \
  --batch-size 1 \
  --warmup 5 \
  --max-new-tokens 64 \
  --output-dir /content/benchmark-results/qwen3

In [ ]:
!python scripts/qwen_benchmark.py run \
  --model qwen2-historical \
  --manifest benchmarks/qwen/samples.jsonl \
  --device cuda:0 \
  --precision float16 \
  --quantization none \
  --batch-size 1 \
  --warmup 5 \
  --max-new-tokens 64 \
  --output-dir /content/benchmark-results/qwen2

In [ ]:
!python scripts/qwen_benchmark.py summarize \
  --predictions /content/benchmark-results/qwen3/predictions.jsonl \
                /content/benchmark-results/qwen2/predictions.jsonl \
  --output /content/benchmark-results/comparison.json

In [ ]:
import json
from pathlib import Path

comparison = json.loads(
    Path('/content/benchmark-results/comparison.json').read_text()
)
for run in comparison['runs']:
    print(
        run['model_label'],
        run['overall']['accuracy'],
        run['overall']['invalid_output_rate'],
    )
print(comparison['paired_comparisons'])

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    '/content/hashformers-issue-78-results',
    'zip',
    '/content/benchmark-results',
)
files.download(archive)